# **Load the Test Dataset**

In [59]:
dataset_path = "/kaggle/input/the-uyghur-voice-cup"
test_df = pd.read_csv(f"{dataset_path}/test.csv")
test_df["filepath"] = test_df["filepath"].apply(lambda x: f"{dataset_path}/{x}")

# **Load the model**

In [60]:
import pandas as pd
import torchaudio
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from tqdm.notebook import tqdm


model_id = "ixxan/whisper-small-uyghur-thugy20"
processor = WhisperProcessor.from_pretrained(model_id)
model = WhisperForConditionalGeneration.from_pretrained(model_id)
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()


WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
        

# **Make prediction on Test Dataset**

In [61]:
predictions = []

for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
    path = row["filepath"]
    waveform, sample_rate = torchaudio.load(path)
    
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    inputs = processor(
        waveform.squeeze().numpy(), 
        sampling_rate=16000, 
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        pred_ids = model.generate(inputs["input_features"])
        transcription = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]

    predictions.append({"ID": row["ID"], "transcription": transcription})

  0%|          | 0/1894 [00:00<?, ?it/s]

# **Generate and clean the submission csv file** 

In [62]:
submission_df = pd.DataFrame(predictions)
submission_df.to_csv("Submission.csv", index=False)

In [64]:
import unicodedata
import re

df = pd.read_csv("Submission.csv")

def clean_text(text):
    fixed = text.encode("latin1", errors="ignore").decode("utf-8", errors="ignore")
    fixed = re.sub(r"[^\w\s]", "", fixed)
    fixed = unicodedata.normalize("NFKC", fixed)
    fixed = fixed.lower().strip()
    return fixed

df["transcription"] = df["transcription"].apply(clean_text)
df.to_csv("submission.csv", index=False)